# EDA 004.03 — Multivariate Analysis

**Visualising three or more variables simultaneously**

## Learning Objectives
By the end of this notebook you will be able to:
- Interpret pair plots to survey all pairwise relationships at once
- Build and annotate correlation heatmaps
- Use facet grids to condition plots on a third variable
- Encode a third or fourth variable via colour, size, and shape
- Read parallel coordinates plots for high-dimensional comparison

---

## Why Multivariate Analysis?

Real datasets have many variables that interact. Univariate and bivariate views can **miss**:
- **Confounding** — a relationship exists only when a third variable is controlled
- **Interaction effects** — variable A predicts B differently depending on C
- **Clusters** — groups of observations that are similar across many dimensions

| Technique | Variables | Best for |
|---|---|---|
| Pair plot | All numerical | Survey all pairwise relationships |
| Correlation heatmap | All numerical | Identify correlated feature pairs |
| Facet grid | 2 numerical + 1-2 categorical | Condition a bivariate plot on groups |
| Colour/size encoding | 2-4 mixed | Add extra dimensions to scatter |
| Parallel coordinates | Many numerical | Compare multi-dim profiles per class |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pandas.plotting import parallel_coordinates

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

penguins = sns.load_dataset('penguins').dropna()
tips     = sns.load_dataset('tips')
iris     = sns.load_dataset('iris')

print('Penguins:', penguins.shape)
penguins.head()

---
## 1 — Pair Plot

**Reference:** [seaborn.pairplot](https://seaborn.pydata.org/generated/seaborn.pairplot.html)

A pair plot (scatter plot matrix) shows:
- **Diagonal** — univariate distribution of each variable (histogram or KDE)
- **Off-diagonal** — bivariate scatter for every pair of variables

Add `hue` to colour points by a categorical variable — immediately reveals clustering.

In [ ]:
# Pair plot — penguins coloured by species
g = sns.pairplot(penguins,
                 vars=['bill_length_mm', 'bill_depth_mm',
                       'flipper_length_mm', 'body_mass_g'],
                 hue='species', diag_kind='kde',
                 plot_kws={'alpha': 0.5, 's': 20})
g.fig.suptitle('Penguin Measurements — Pair Plot by Species',
               y=1.01, fontweight='bold')
plt.show()

---
## 2 — Correlation Heatmap

**Reference:** [seaborn.heatmap](https://seaborn.pydata.org/generated/seaborn.heatmap.html) | [pandas.DataFrame.corr](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corr.html)

A correlation heatmap shows the **Pearson correlation coefficient** (−1 to +1) between every pair of numerical variables.

Tips:
- Use a **diverging colormap** (e.g. `coolwarm`, `RdBu_r`) centred on 0
- Set `annot=True` to display the actual values
- Mask the upper triangle to reduce clutter (`mask=np.triu(...)`)
- High correlation (|r| > 0.8) can indicate **multicollinearity** — a problem for linear models

In [ ]:
num_cols = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
corr_matrix = penguins[num_cols].corr()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full heatmap
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, center=0,
            square=True, linewidths=0.5, ax=axes[0])
axes[0].set_title('Full Correlation Matrix')

# Lower triangle only
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', vmin=-1, vmax=1, center=0,
            square=True, linewidths=0.5, ax=axes[1])
axes[1].set_title('Lower Triangle (less redundancy)')

plt.tight_layout()
plt.show()

---
## 3 — Facet Grid

**Reference:** [seaborn.FacetGrid](https://seaborn.pydata.org/generated/seaborn.FacetGrid.html) | [seaborn.catplot](https://seaborn.pydata.org/generated/seaborn.catplot.html)

A facet grid creates a **grid of subplots**, each showing the same bivariate plot for a different subset of the data (conditioned on one or two categorical variables).

This makes it easy to ask: *"Does this relationship hold for all groups?"*

In [ ]:
# catplot — facet by sex, show tip distribution by day
g = sns.catplot(data=tips, x='day', y='tip', col='sex',
                kind='box', height=4, aspect=1.1,
                palette='Set2', order=['Thur', 'Fri', 'Sat', 'Sun'])
g.set_titles('Sex: {col_name}')
g.fig.suptitle('Tip by Day, Faceted by Sex', y=1.02, fontweight='bold')
plt.show()

In [ ]:
# FacetGrid — scatter faceted by island (row) and sex (col)
g = sns.FacetGrid(penguins, row='island', col='sex',
                  hue='species', height=2.5, aspect=1.3)
g.map(sns.scatterplot, 'bill_length_mm', 'body_mass_g', alpha=0.6, s=25)
g.add_legend()
g.set_titles('{row_name} | {col_name}')
g.fig.suptitle('Bill Length vs Body Mass — Faceted by Island × Sex',
               y=1.01, fontweight='bold')
plt.show()

---
## 4 — Encoding a 3rd and 4th Variable via Colour and Size

**Reference:** [seaborn.scatterplot](https://seaborn.pydata.org/generated/seaborn.scatterplot.html)

In a 2D scatter plot you can encode additional variables through:

| Channel | Use for |
|---|---|
| `hue` | Categorical or continuous variable |
| `size` | Continuous quantity (e.g. population, score) |
| `style` | Categorical variable (shape markers) |

> **Limit:** Beyond 4 variables, plots become hard to read. Consider dimensionality reduction (PCA) instead.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

# x, y = two numerical; hue = species; size = body_mass; style = sex
sns.scatterplot(data=penguins,
                x='flipper_length_mm',
                y='bill_length_mm',
                hue='species',
                size='body_mass_g',
                style='sex',
                sizes=(20, 200),
                alpha=0.7,
                palette='deep',
                ax=ax)
ax.set_title('4 variables in one plot: flipper vs bill (hue=species, size=mass, style=sex)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

---
## 5 — Parallel Coordinates

**Reference:** [pandas.plotting.parallel_coordinates](https://pandas.pydata.org/docs/reference/api/pandas.plotting.parallel_coordinates.html)

Parallel coordinates visualise **multi-dimensional profiles** for each observation:
- Each **vertical axis** is one feature (normalised to the same scale)
- Each **line** is one observation — coloured by class
- Lines that cluster together indicate groups with similar profiles

Best for: comparing how different classes differ across many numerical features.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Normalise for fair visual comparison
iris_num_cols = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
iris_scaled = iris.copy()
iris_scaled[iris_num_cols] = MinMaxScaler().fit_transform(iris[iris_num_cols])

fig, ax = plt.subplots(figsize=(10, 5))
parallel_coordinates(iris_scaled, class_column='species',
                     color=['#E74C3C', '#27AE60', '#2980B9'],
                     alpha=0.4, ax=ax)
ax.set_title('Parallel Coordinates — Iris Species (normalised features)')
ax.set_ylabel('Normalised value (0–1)')
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

---
## Key Takeaways

- Pair plots give a rapid survey of all pairwise numerical relationships — start here
- Correlation heatmaps quantify those relationships and highlight multicollinearity
- Facet grids check whether a relationship holds across different groups
- Colour + size encoding adds extra variables to a scatter plot (max ~4 channels)
- Parallel coordinates are powerful for comparing multi-dimensional class profiles

---
## Further Reading

- [Seaborn — Statistical Data Visualization](https://seaborn.pydata.org/tutorial.html)
- [Fundamentals of Data Visualization (free)](https://clauswilke.com/dataviz/) — Ch. 12-21
- [Kaggle — Data Visualization Course](https://www.kaggle.com/learn/data-visualization)
- [Pandas Visualization](https://pandas.pydata.org/docs/user_guide/visualization.html)
- [Plotly for Interactive Multivariate Plots](https://plotly.com/python/parallel-coordinates-plot/)